In [ ]:
import random
import matplotlib.pyplot as plt
import networkx as nx
from dataclasses import dataclass, field
from typing import Optional

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

# True  -> genera un árbol aleatorio
# False -> utiliza el árbol escrito manualmente
generar_aleatorio = False


# ------------------------------------------------------------
# OPCIÓN 1: Árbol escrito manualmente
# ------------------------------------------------------------

arbol = [
    [3, -5, 2],
    [-5, -7, 4],
    [-9, 6, -8]
]


# ------------------------------------------------------------
# OPCIÓN 2: Generación aleatoria
# ------------------------------------------------------------

niveles = 3
numero_hojas = 9

# Rango de valores para las hojas
valor_minimo = -10
valor_maximo = 10

# Semilla para poder reproducir el mismo árbol
semilla = 42

In [ ]:
# ============================================================
# GENERACIÓN DE ÁRBOLES ALEATORIOS
# ============================================================

def generar_arbol_aleatorio(
    niveles,
    numero_hojas,
    valor_minimo=-10,
    valor_maximo=10,
    semilla=None
):
    """
    Genera un árbol representado mediante listas anidadas.

    niveles:
        Número de niveles del árbol incluyendo raíz y hojas.

    numero_hojas:
        Cantidad exacta de nodos hoja.

    valor_minimo / valor_maximo:
        Rango de valores para las hojas.

    semilla:
        Permite reproducir el mismo árbol.
    """

    if semilla is not None:
        random.seed(semilla)

    if niveles < 2:
        raise ValueError("El árbol debe tener al menos 2 niveles.")

    if numero_hojas < 1:
        raise ValueError("Debe existir al menos un nodo hoja.")

    def construir(nivel, hojas):
        # Último nivel: crear las hojas
        if nivel == niveles - 1:
            return [
                random.randint(valor_minimo, valor_maximo)
                for _ in range(hojas)
            ]

        # Cantidad mínima y máxima de hojas por hijo
        niveles_restantes = niveles - nivel - 1

        # Permitir entre 2 y 4 hijos por nodo cuando sea posible
        max_hijos = min(4, hojas)

        if max_hijos < 1:
            max_hijos = 1

        # Elegir cantidad de hijos
        if hojas >= 2:
            cantidad_hijos = random.randint(2, max_hijos)
        else:
            cantidad_hijos = 1

        # Distribuir las hojas entre los hijos
        distribucion = [1] * cantidad_hijos

        restantes = hojas - cantidad_hijos

        while restantes > 0:
            indice = random.randrange(cantidad_hijos)
            distribucion[indice] += 1
            restantes -= 1

        return [
            construir(nivel + 1, cantidad_hojas)
            for cantidad_hojas in distribucion
        ]

    return construir(0, numero_hojas)

In [ ]:
# ============================================================
# PREPARAR ÁRBOL
# ============================================================

if generar_aleatorio:
    arbol = generar_arbol_aleatorio(
        niveles=niveles,
        numero_hojas=numero_hojas,
        valor_minimo=valor_minimo,
        valor_maximo=valor_maximo,
        semilla=semilla
    )

print("Árbol utilizado:")
print(arbol)

In [ ]:
# ============================================================
# ESTRUCTURA DE NODOS
# ============================================================

@dataclass
class Nodo:
    id: int
    valor: Optional[int] = None
    hijos: list = field(default_factory=list)

    nivel: int = 0

    # Información utilizada por Alfa-Beta
    tipo: str = ""
    visitado: bool = False
    podado: bool = False
    optimo: bool = False

    alpha: Optional[float] = None
    beta: Optional[float] = None


contador_nodos = 0


def crear_nodos(estructura, nivel=0):
    """
    Convierte una lista anidada en una estructura de objetos Nodo.
    """

    global contador_nodos

    nodo = Nodo(
        id=contador_nodos,
        nivel=nivel
    )

    contador_nodos += 1

    # Si es una hoja
    if isinstance(estructura, (int, float)):
        nodo.valor = estructura
        return nodo

    # Si es una lista, crear los hijos
    for elemento in estructura:
        hijo = crear_nodos(elemento, nivel + 1)
        nodo.hijos.append(hijo)

    return nodo

In [ ]:
# ============================================================
# CONSTRUIR ÁRBOL DE NODOS
# ============================================================

contador_nodos = 0

raiz = crear_nodos(arbol)

print("Árbol convertido correctamente.")
print(f"Número total de nodos: {contador_nodos}")

In [ ]:
# ============================================================
# ASIGNAR MAX / MIN
# ============================================================

def asignar_tipos(nodo, es_max=True):
    """
    Alterna entre MAX y MIN en cada nivel.
    """

    nodo.tipo = "MAX" if es_max else "MIN"

    for hijo in nodo.hijos:
        asignar_tipos(hijo, not es_max)


asignar_tipos(raiz, True)

In [ ]:
# ============================================================
# MINIMAX CON PODA ALFA-BETA
# ============================================================

podas = []
camino_optimo = []


def marcar_subarbol_podado(nodo):
    """
    Marca un nodo y todos sus descendientes como podados.
    """

    nodo.podado = True

    for hijo in nodo.hijos:
        marcar_subarbol_podado(hijo)


def minimax_alfa_beta(nodo, alpha, beta, profundidad=0):
    """
    Algoritmo Minimax con Poda Alfa-Beta.

    MAX intenta maximizar el valor.
    MIN intenta minimizar el valor.
    """

    nodo.visitado = True
    nodo.alpha = alpha
    nodo.beta = beta

    # --------------------------------------------------------
    # CASO BASE: nodo hoja
    # --------------------------------------------------------

    if not nodo.hijos:
        return nodo.valor

    # --------------------------------------------------------
    # NODO MAX
    # --------------------------------------------------------

    if nodo.tipo == "MAX":

        mejor_valor = float("-inf")

        for i, hijo in enumerate(nodo.hijos):

            valor = minimax_alfa_beta(
                hijo,
                alpha,
                beta,
                profundidad + 1
            )

            mejor_valor = max(mejor_valor, valor)

            alpha = max(alpha, mejor_valor)

            nodo.alpha = alpha

            # Poda Beta
            if beta <= alpha:

                for restante in nodo.hijos[i + 1:]:
                    marcar_subarbol_podado(restante)

                    podas.append({
                        "nodo": nodo.id,
                        "tipo": "Beta",
                        "mensaje": f"Se poda la rama a partir del nodo {restante.id}"
                    })

                break

        nodo.valor = mejor_valor

        return mejor_valor

    # --------------------------------------------------------
    # NODO MIN
    # --------------------------------------------------------

    else:

        mejor_valor = float("inf")

        for i, hijo in enumerate(nodo.hijos):

            valor = minimax_alfa_beta(
                hijo,
                alpha,
                beta,
                profundidad + 1
            )

            mejor_valor = min(mejor_valor, valor)

            beta = min(beta, mejor_valor)

            nodo.beta = beta

            # Poda Alfa
            if beta <= alpha:

                for restante in nodo.hijos[i + 1:]:
                    marcar_subarbol_podado(restante)

                    podas.append({
                        "nodo": nodo.id,
                        "tipo": "Alfa",
                        "mensaje": f"Se poda la rama a partir del nodo {restante.id}"
                    })

                break

        nodo.valor = mejor_valor

        return mejor_valor

In [ ]:
# ============================================================
# EJECUTAR ALFA-BETA
# ============================================================

valor_optimo = minimax_alfa_beta(
    raiz,
    float("-inf"),
    float("inf")
)

print("=" * 50)
print("RESULTADO DEL ALGORITMO")
print("=" * 50)

print(f"Valor óptimo: {valor_optimo}")
print(f"Nodos totales: {contador_nodos}")
print(f"Nodos podados: {sum(1 for nodo in [] )}")
print(f"Cantidad de podas realizadas: {len(podas)}")

In [ ]:
# ============================================================
# OBTENER INFORMACIÓN DE LOS NODOS
# ============================================================

def obtener_todos_los_nodos(nodo):
    """
    Devuelve una lista con todos los nodos del árbol.
    """

    resultado = [nodo]

    for hijo in nodo.hijos:
        resultado.extend(obtener_todos_los_nodos(hijo))

    return resultado


todos_los_nodos = obtener_todos_los_nodos(raiz)

nodos_visitados = [
    nodo for nodo in todos_los_nodos
    if nodo.visitado
]

nodos_podados = [
    nodo for nodo in todos_los_nodos
    if nodo.podado
]

hojas = [
    nodo for nodo in todos_los_nodos
    if not nodo.hijos
]


print("=" * 50)
print("ESTADÍSTICAS")
print("=" * 50)

print(f"Nodos totales: {len(todos_los_nodos)}")
print(f"Nodos hoja: {len(hojas)}")
print(f"Nodos visitados: {len(nodos_visitados)}")
print(f"Nodos podados: {len(nodos_podados)}")
print(f"Podas realizadas: {len(podas)}")
print(f"Valor óptimo: {valor_optimo}")

In [ ]:
# ============================================================
# CAMINO ÓPTIMO
# ============================================================

def encontrar_camino_optimo(nodo, objetivo):
    """
    Encuentra un camino desde la raíz hasta una hoja que
    produzca el valor óptimo.
    """

    if not nodo.hijos:
        if nodo.valor == objetivo:
            nodo.optimo = True
            return [nodo]

        return None

    # Buscar hijos cuyo valor coincida con el valor del nodo
    candidatos = [
        hijo for hijo in nodo.hijos
        if not hijo.podado and hijo.valor == nodo.valor
    ]

    for hijo in candidatos:

        resultado = encontrar_camino_optimo(
            hijo,
            objetivo
        )

        if resultado is not None:
            nodo.optimo = True
            return [nodo] + resultado

    return None


camino = encontrar_camino_optimo(
    raiz,
    valor_optimo
)

camino_optimo = camino if camino else []


print("=" * 50)
print("SECUENCIA ÓPTIMA")
print("=" * 50)

for nodo in camino_optimo:
    print(
        f"Nodo {nodo.id} | "
        f"Nivel {nodo.nivel} | "
        f"{nodo.tipo} | "
        f"Valor = {nodo.valor}"
    )

In [ ]:
# ============================================================
# RAMAS PODADAS
# ============================================================

print("=" * 50)
print("RAMAS PODADAS")
print("=" * 50)

if not podas:
    print("No se realizaron podas Alfa-Beta.")
else:

    for i, poda in enumerate(podas, 1):

        print(
            f"{i}. Nodo {poda['nodo']} "
            f"-> Poda {poda['tipo']} "
            f"-> {poda['mensaje']}"
        )

In [ ]:
# ============================================================
# CONSTRUIR GRAFO
# ============================================================

def construir_grafo(nodo, grafo=None, padre=None):
    """
    Convierte el árbol de nodos en un grafo de NetworkX.
    """

    if grafo is None:
        grafo = nx.DiGraph()

    grafo.add_node(
        nodo.id,
        nivel=nodo.nivel,
        valor=nodo.valor,
        tipo=nodo.tipo,
        podado=nodo.podado,
        optimo=nodo.optimo,
        visitado=nodo.visitado
    )

    if padre is not None:
        grafo.add_edge(padre.id, nodo.id)

    for hijo in nodo.hijos:
        construir_grafo(hijo, grafo, nodo)

    return grafo


G = construir_grafo(raiz)

print(
    f"Grafo creado con {G.number_of_nodes()} nodos "
    f"y {G.number_of_edges()} conexiones."
)

In [ ]:
# ============================================================
# VISUALIZACIÓN DEL ÁRBOL
# ============================================================

def posiciones_arbol(G):
    """
    Genera posiciones manuales para que el árbol se visualice
    de arriba hacia abajo.
    """

    niveles = {}

    for nodo, datos in G.nodes(data=True):
        nivel = datos["nivel"]

        if nivel not in niveles:
            niveles[nivel] = []

        niveles[nivel].append(nodo)

    posiciones = {}

    max_nodos_nivel = max(
        len(nodos)
        for nodos in niveles.values()
    )

    for nivel, nodos in niveles.items():

        cantidad = len(nodos)

        for i, nodo in enumerate(nodos):

            if cantidad == 1:
                x = 0
            else:
                x = (
                    i - (cantidad - 1) / 2
                ) / max(1, cantidad - 1)

            y = -nivel

            posiciones[nodo] = (x, y)

    return posiciones


posiciones = posiciones_arbol(G)


plt.figure(figsize=(18, 10))

# ------------------------------------------------------------
# Separar tipos de nodos
# ------------------------------------------------------------

nodos_normales = []
nodos_visitados_no_optimos = []
nodos_podados = []
nodos_optimos = []

for nodo, datos in G.nodes(data=True):

    if datos["podado"]:
        nodos_podados.append(nodo)

    elif datos["optimo"]:
        nodos_optimos.append(nodo)

    elif datos["visitado"]:
        nodos_visitados_no_optimos.append(nodo)

    else:
        nodos_normales.append(nodo)


# ------------------------------------------------------------
# Dibujar aristas
# ------------------------------------------------------------

for padre, hijo in G.edges():

    datos_hijo = G.nodes[hijo]

    if datos_hijo["podado"]:
        estilo = "--"
        ancho = 1.5

    elif datos_hijo["optimo"]:
        estilo = "-"
        ancho = 4

    else:
        estilo = "-"
        ancho = 1

    nx.draw_networkx_edges(
        G,
        posiciones,
        edgelist=[(padre, hijo)],
        style=estilo,
        width=ancho
    )


# ------------------------------------------------------------
# Dibujar nodos
# ------------------------------------------------------------

nx.draw_networkx_nodes(
    G,
    posiciones,
    nodelist=nodos_normales,
    node_size=900
)

nx.draw_networkx_nodes(
    G,
    posiciones,
    nodelist=nodos_visitados_no_optimos,
    node_size=900
)

nx.draw_networkx_nodes(
    G,
    posiciones,
    nodelist=nodos_podados,
    node_size=900,
    node_shape="X"
)

nx.draw_networkx_nodes(
    G,
    posiciones,
    nodelist=nodos_optimos,
    node_size=1100,
    node_shape="*"
)


# ------------------------------------------------------------
# Etiquetas
# ------------------------------------------------------------

etiquetas = {}

for nodo, datos in G.nodes(data=True):

    if datos["valor"] is not None:
        valor = datos["valor"]
    else:
        valor = "?"

    etiquetas[nodo] = (
        f"{datos['tipo']}\n"
        f"ID: {nodo}\n"
        f"V: {valor}"
    )


nx.draw_networkx_labels(
    G,
    posiciones,
    labels=etiquetas,
    font_size=8
)


# ------------------------------------------------------------
# Título
# ------------------------------------------------------------

plt.title(
    "Minimax con Poda Alfa-Beta\n"
    "Secuencia óptima y ramas podadas",
    fontsize=16
)

plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# LEYENDA
# ============================================================

print()
print("=" * 60)
print("LEYENDA DE LA FIGURA")
print("=" * 60)

print("★  Nodo perteneciente a la secuencia óptima")
print("X  Nodo perteneciente a una rama podada")
print("○  Nodo evaluado o perteneciente al árbol")
print("-  Rama normal")
print("-- Rama podada")
print()
print("MAX → jugador que busca maximizar el valor")
print("MIN → jugador que busca minimizar el valor")

In [ ]:
# ============================================================
# RESUMEN FINAL
# ============================================================

print()
print("=" * 60)
print("RESUMEN DEL ALGORITMO ALFA-BETA")
print("=" * 60)

print(f"Cantidad de niveles: {max(n.nivel for n in todos_los_nodos) + 1}")
print(f"Cantidad de nodos hoja: {len(hojas)}")
print(f"Cantidad de nodos totales: {len(todos_los_nodos)}")
print(f"Cantidad de nodos visitados: {len(nodos_visitados)}")
print(f"Cantidad de nodos podados: {len(nodos_podados)}")
print(f"Cantidad de podas realizadas: {len(podas)}")
print(f"Valor óptimo encontrado: {valor_optimo}")

print("\nSecuencia óptima:")

for i, nodo in enumerate(camino_optimo):

    if i == 0:
        print(f"  Inicio → Nodo {nodo.id} ({nodo.tipo})")

    else:
        print(
            f"  ↓ Nodo {nodo.id} "
            f"({nodo.tipo}) → Valor {nodo.valor}"
        )

print("\nProceso finalizado correctamente.")